# Annualized volatility predictions

Author: Pete King

In our analysis, we selected a Long Short-Term Memory (LSTM) model using only a few input features as the best performing model according to our evaluation criteria.

Here we convert the model's predictions of realized volatility to annualized volatility.  For each ETF, we provide two values:

 - An average annualized volatility over the test set (from 61 weekly predictions)
 - Annualized volatility for the latest available prediction in the test set

These values are based on only one training and forecasting run of the model.  Each run of the forecasting script (model_testing_volatility.py) takes ~21 minutes to complete.  Once more runs are completed, we can an average of those runs, which should be a better overall estimate.

In [1]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import altair as alt

In [2]:
with open('volatility_results.json') as f:
    results = json.load(f)

In [3]:
etfs = list(results['1'].keys())
etfs

['BIL',
 'BND',
 'GLD',
 'HYG',
 'IEF',
 'IWM',
 'LQD',
 'QQQ',
 'SPY',
 'TIP',
 'TLT',
 'XLB',
 'XLE',
 'XLF',
 'XLI',
 'XLK',
 'XLP',
 'XLRE',
 'XLU',
 'XLV',
 'XLY']

In [4]:
print('Annualized volatility predictions (avg over test set)\n---')
avg_vols = []
for etf in etfs:
    real_vol_avg = np.mean(results['1'][etf]['LSTM']['limited']['test']['y_pred'])
    ann_vol_avg =  real_vol_avg * np.sqrt(252) * 100
    avg_vols.append(ann_vol_avg)
    print(f'{etf}: {ann_vol_avg:.2f}')

Annualized volatility predictions (avg over test set)
---
BIL: 0.43
BND: 5.44
GLD: 18.31
HYG: 4.93
IEF: 6.02
IWM: 25.83
LQD: 7.37
QQQ: 23.73
SPY: 16.15
TIP: 4.83
TLT: 18.66
XLB: 19.95
XLE: 25.63
XLF: 22.50
XLI: 18.02
XLK: 25.24
XLP: 10.30
XLRE: 18.71
XLU: 15.98
XLV: 15.58
XLY: 25.73


In [5]:
print('Annualized volatility predictions (last prediction in test set)\n---')
latest_vols = []
for etf in etfs:
    real_vol_latest = results['1'][etf]['LSTM']['limited']['test']['y_pred'][-1]
    ann_vol_latest =  real_vol_latest * np.sqrt(252) * 100
    latest_vols.append(ann_vol_latest)
    print(f'{etf}: {ann_vol_latest:.2f}')

Annualized volatility predictions (last prediction in test set)
---
BIL: 0.37
BND: 4.50
GLD: 15.19
HYG: 3.68
IEF: 5.00
IWM: 24.93
LQD: 6.25
QQQ: 21.41
SPY: 13.39
TIP: 4.36
TLT: 17.39
XLB: 14.95
XLE: 27.63
XLF: 19.68
XLI: 15.90
XLK: 22.83
XLP: 9.77
XLRE: 15.61
XLU: 16.45
XLV: 15.77
XLY: 23.73


In [14]:
df = pd.DataFrame(
    [
        pd.Series(avg_vols, name='Avg', index=etfs),
        pd.Series(latest_vols, name='Latest', index=etfs)
    ]
).T

In [15]:
df

,Avg,Latest
BIL,0.430734,0.370293
BND,5.439653,4.496448
GLD,18.306261,15.193564
HYG,4.933129,3.684668
IEF,6.016238,5.000053
IWM,25.834981,24.926250
LQD,7.366070,6.254593
QQQ,23.727153,21.413045
SPY,16.147985,13.391998
TIP,4.832525,4.355256


In [17]:
df.to_csv('annualized_volatility_predictions.csv', index_label='ETF')